### Setup

In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(scales)
  library(patchwork)
  library(ggtext)
  library(ggh4x)
  library(cowplot)
})

BASE_SIZE <- 16

# ## Paths
data_dir <- "../data"
out_dir  <- "../figs"

out_main <- file.path(out_dir, "Fig5.pdf")
out_supp <- file.path(out_dir, "ED_Fig5.pdf")

### Colours, labels and helpers

In [2]:
# ## Colours
ab_cols <- c("0" = "#BBBBBC", "5" = "#AD8CB999", "50" = "#AD8CB9", "500" = "#6D2682")
col_amp    <- "#6D2682"   # purple
col_growth <- "#2BBFD9"   # blue-green

# Background shading
bg_base_fill    <- "white"
bg_present_fill <- "#D0D0D0"
bg_legend_label <- "Primed <i>A. caviae</i> present"

# ## Species colours / labels
hambi_phylo_codes2 <- c(
  "HAMBI_1988","HAMBI_3031",
  "HAMBI_1896",
  "HAMBI_3237","HAMBI_0105","HAMBI_2443","HAMBI_1842","HAMBI_0262",
  "HAMBI_2160","HAMBI_0403","HAMBI_2164","HAMBI_2494","HAMBI_2159",
  "HAMBI_0097","HAMBI_2792","HAMBI_1292","HAMBI_1279","HAMBI_1299","HAMBI_1287",
  "HAMBI_1972","HAMBI_1977","HAMBI_0006","HAMBI_2659"
)

colorpalette <- c(
  "#EA7F40","#EAB0F6","#834F16","#1649F7","#FDD21C","#8200AA","#0DF8EB",
  "#FD0D7A","#6A9940","#82BAFE","#FA2AD9","#FD83B5","#FED8BE","#2AF622",
  "#BEDDE4","#C4EF35","#42F19F","#DC16FC","#70566A","#F6533B","#164F85",
  "#22917A","#BA3249"
)

stopifnot(length(hambi_phylo_codes2) == length(colorpalette))
species_colors <- setNames(colorpalette, hambi_phylo_codes2)

species_labels <- c(
  "HAMBI_0006" = "Pseudomonas putida\nHAMBI 6",
  "HAMBI_0097" = "Acinetobacter johnsonii\nHAMBI 97",
  "HAMBI_0105" = "Agrobacterium tumefaciens\nHAMBI 105",
  "HAMBI_0262" = "Brevundimonas bullata\nHAMBI 262",
  "HAMBI_0403" = "Comamonas testosteroni\nHAMBI 403",
  "HAMBI_1279" = "Hafnia alvei\nHAMBI 1279",
  "HAMBI_1287" = "Citrobacter koseri\nHAMBI 1287",
  "HAMBI_1292" = "Morganella morganii\nHAMBI 1292",
  "HAMBI_1299" = "Kluyvera intermedia\nHAMBI 1299",
  "HAMBI_1842" = "Sphingobium yanoikuyae\nHAMBI 1842",
  "HAMBI_1896" = "Sphingobacterium spiritivorum\nHAMBI 1896",
  "HAMBI_1972" = "Aeromonas caviae\nHAMBI 1972",
  "HAMBI_1977" = "Pseudomonas chlororaphis\nHAMBI 1977",
  "HAMBI_1988" = "Chitinophaga sancti\nHAMBI 1988",
  "HAMBI_2159" = "Tricknia caryophylli\nHAMBI 2159",
  "HAMBI_2160" = "Bordetella avium\nHAMBI 2160",
  "HAMBI_2164" = "Cupriavidus oxalaticus\nHAMBI 2164",
  "HAMBI_2443" = "Paracoccus denitrificans\nHAMBI 2443",
  "HAMBI_2494" = "Paraburkholderia kururiensis\nHAMBI 2494",
  "HAMBI_2659" = "Stenotrophomonas maltophilia\nHAMBI 2659",
  "HAMBI_2792" = "Moraxella canis\nHAMBI 2792",
  "HAMBI_3031" = "Niabella yanshanensis\nHAMBI 3031",
  "HAMBI_3237" = "Microvirga lotononidis\nHAMBI 3237"
)

species_labels_md <- species_labels %>%
  gsub("\n", " ", .) %>%
  gsub("^([A-Za-z]+\\s+[a-z]+)", "<i>\\1</i>", .)
species_labels_1line <- setNames(species_labels_md, names(species_labels))

# ## IDs / labels
id_acav <- "HAMBI_1972"
id_1977 <- "HAMBI_1977"
id_0403 <- "HAMBI_0403"
degrader_ids <- c(id_acav, id_1977, id_0403)

absent_levels <- c("None", "403", "1977", "1972", "All3")
valid_times <- c(6, 12, 24)

absent_labels_md <- c(
  "None" = "All species present",
  "403"  = "Ampicillin degrader<br><i>C. testosteroni</i> dropped",
  "1977" = "Ampicillin degrader<br><i>P. chlororaphis</i> dropped",
  "1972" = "Ampicillin degrader<br><i>A. caviae</i> dropped",
  "All3" = "All degraders dropped"
)

evo_labels_2line <- c(
  "ANC" = "Ancestral community",
  "EVO" = "Full primed"
)

# markdown labels for legends
lab_Camp <- "C<sub>amp</sub>"
lab_acav_prop <- "<i>A. caviae</i> proportion"
title_prepulse_Camp <- "Pre-pulse priming<br>C<sub>amp</sub> (µg mL⁻¹)"

# ## Helpers
fix_names <- function(df) {
  cn <- colnames(df)
  cn[is.na(cn) | cn == ""] <- paste0("X", which(is.na(cn) | cn == ""))
  colnames(df) <- make.unique(cn, sep = ".")
  df
}

mean_ci_95 <- function(x) {
  x <- x[is.finite(x)]
  n <- length(x)
  if (n == 0) return(tibble(mean = NA_real_, lo = NA_real_, hi = NA_real_))
  m <- mean(x)
  se <- sd(x) / sqrt(n)
  tibble(mean = m, lo = m - 1.96 * se, hi = m + 1.96 * se)
}

### Panel A

In [3]:
df_key_species <- read.table(
  file.path(data_dir, "AMP_key_species.txt"),
  header = TRUE, sep = "\t", dec = ","
)

df4ks <- df_key_species %>%
  filter(Species %in% c("HAMBI_403", "HAMBI_1972", "HAMBI_1977")) %>%
  mutate(
    Evo = factor(Evo, levels = c("ANC", "EVO"), labels = c("Anc.", "Primed")),
    Species = factor(
      Species,
      levels = c("HAMBI_1972", "HAMBI_403", "HAMBI_1977"),
      labels = c(
        "italic(Aeromonas~caviae)",
        "italic(Comamonas~testosteroni)",
        "italic(Pseudomonas~chlororaphis)"
      )
    ),
    FillEvo = Evo,
    Timeh = as.numeric(Timeh),
    AMP_conc = as.numeric(AMP_conc),
    OD = as.numeric(OD)
  )

scaleFactor2 <- max(df4ks$AMP_conc, na.rm = TRUE) / max(df4ks$OD, na.rm = TRUE)
df4ks_long <- df4ks %>% mutate(C_amp = AMP_conc, Biomass = OD)

pA_base <- ggplot(df4ks_long, aes(Timeh, group = Evo)) +
  stat_summary(
    aes(y = C_amp, colour = "Ampicillin", linetype = Evo),
    fun = mean, geom = "line", linewidth = 1.0
  ) +
  stat_summary(
    aes(y = C_amp, colour = "Ampicillin", shape = Evo, fill = FillEvo),
    fun = mean, geom = "point", size = 4.2, stroke = 1.2
  ) +
  geom_jitter(
    aes(y = C_amp, colour = "Ampicillin", shape = Evo, fill = FillEvo),
    position = position_jitter(width = 0.35, height = 0),
    alpha = 0.20, size = 2.8, stroke = 0.9
  ) +
  stat_summary(
    aes(y = Biomass * scaleFactor2, colour = "Biomass", linetype = Evo),
    fun = mean, geom = "line", linewidth = 1.0
  ) +
  stat_summary(
    aes(y = Biomass * scaleFactor2, colour = "Biomass", shape = Evo, fill = FillEvo),
    fun = mean, geom = "point", size = 4.2, stroke = 1.2
  ) +
  geom_jitter(
    aes(y = Biomass * scaleFactor2, colour = "Biomass", shape = Evo, fill = FillEvo),
    position = position_jitter(width = 0.35, height = 0),
    alpha = 0.20, size = 2.8, stroke = 0.9
  ) +
  scale_x_continuous(breaks = c(0, 6, 12, 24)) +
  scale_colour_manual(
    values = c("Ampicillin" = col_amp, "Biomass" = col_growth),
    breaks = c("Ampicillin", "Biomass"),
    labels = c(lab_Camp, "Biomass"),
    name = NULL
  ) +
  scale_shape_manual(values = c("Anc." = 21, "Primed" = 24), name = NULL) +
  scale_fill_manual(values = c("Anc." = "white", "Primed" = "grey15"), name = NULL) +
  scale_linetype_manual(values = c("Anc." = "solid", "Primed" = "dashed"), name = NULL) +
  scale_y_continuous(
    name = expression(bold(C[amp]~"("*mu*g~mL^-1*")")),
    sec.axis = sec_axis(~ . / scaleFactor2, name = expression(bold(Biomass~"("*OD[600]*")")))
  ) +
  labs(x = "Time (h)") +
  facet_grid(. ~ Species, scales = "free_y", labeller = label_parsed) +
  theme_bw(base_size = BASE_SIZE) +
  theme(
    panel.grid       = element_blank(),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    panel.border     = element_rect(colour = "black", fill = NA, linewidth = 0.8),
    axis.title.x     = element_text(face = "bold", size = BASE_SIZE),
    axis.title.y     = element_text(face = "bold", size = BASE_SIZE),
    axis.title.y.right = element_text(face = "bold", size = BASE_SIZE, margin = margin(l = 4)),
    axis.text        = element_text(size = BASE_SIZE),
    axis.text.y.right = element_text(size = BASE_SIZE, margin = margin(l = 2)),
    strip.background = element_blank(),
    strip.text       = element_text(size = BASE_SIZE),
    legend.position  = "top",
    legend.text      = ggtext::element_markdown(size = BASE_SIZE),
    legend.title     = element_blank(),
    plot.margin      = margin(4, 2, 6, 10)
  )

legend_top <- cowplot::get_legend(
  pA_base + theme(legend.position = "top")
)
legend_top_patch <- cowplot::ggdraw(legend_top)

pA <- pA_base +
  theme(
    legend.position = "none",
    plot.tag = element_text(size = BASE_SIZE + 2, face = "bold")
  ) +
  labs(tag = "A")


### Panel B

In [4]:
species_loo <- read_tsv(file.path(data_dir, "species_abundance_loo.txt"), show_col_types = FALSE) %>% fix_names()
meta_loo    <- read_tsv(file.path(data_dir, "metadata_loo.txt"),          show_col_types = FALSE) %>% fix_names()
amp_loo     <- read_tsv(file.path(data_dir, "AMP_conc_loo.txt"),          show_col_types = FALSE) %>% fix_names()

meta2 <- meta_loo %>%
  mutate(Absent_norm = recode(Absent, "All" = "All3")) %>%
  filter(
    EVO %in% c("ANC", "EVO"),
    !(row %in% c("Cont", "Neg", "CC")),
    ampicillin_ug_ml == 500
  ) %>%
  mutate(
    Absent_norm = factor(Absent_norm, levels = absent_levels),
    replicate = as.character(replicate)
  )

counts_filtered <- species_loo %>%
  inner_join(meta2 %>% select(sample, Timepoint, EVO, replicate, Absent_norm), by = "sample") %>%
  filter(Timepoint %in% valid_times) %>%
  filter(!(
    (Absent_norm == "1972" & strainID == id_acav) |
      (Absent_norm == "1977" & strainID == id_1977) |
      (Absent_norm == "403"  & strainID == id_0403) |
      (Absent_norm == "All3" & strainID %in% degrader_ids)
  ))

amp2 <- amp_loo %>%
  filter(Evo %in% c("ANC", "EVO"), Time_h %in% valid_times) %>%
  transmute(
    EVO = Evo,
    Absent_norm = recode(Missing_species, "none" = "None"),
    replicate = as.character(Rep),
    Timepoint = Time_h,
    C_amp
  ) %>%
  mutate(Absent_norm = factor(Absent_norm, levels = absent_levels))

conc_max <- max(amp2$C_amp, na.rm = TRUE)
amp2 <- amp2 %>% mutate(amp_scaled = C_amp / conc_max)

# ============================================================
# PANEL B
# ============================================================

bg_df_B <- expand_grid(
  EVO = factor(c("ANC", "EVO"), levels = c("ANC", "EVO")),
  Absent_norm = factor(absent_levels, levels = absent_levels)
) %>%
  mutate(present = (EVO == "EVO" & Absent_norm %in% c("None", "403", "1977")))

bg_tile_B <- bg_df_B %>%
  filter(present) %>%
  mutate(Timepoint = valid_times[1], y = 0.5)

bg_legend_key <- tibble(
  Timepoint = valid_times[1],
  y = 0.02,
  bg_key = bg_legend_label
)

aero_by_sample <- counts_filtered %>%
  group_by(sample, EVO, Absent_norm, replicate, Timepoint) %>%
  summarise(
    total = sum(f_correct, na.rm = TRUE),
    acav  = sum(f_correct[strainID == id_acav], na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(acav_prop = acav / pmax(total, 1e-12))

acav_sum <- aero_by_sample %>%
  group_by(EVO, Absent_norm, Timepoint) %>%
  summarise(mean_ci_95(acav_prop), .groups = "drop") %>%
  transmute(
    EVO, Absent_norm, Timepoint,
    Series = lab_acav_prop,
    y = mean, ylo = pmax(lo, 0), yhi = pmin(hi, 1)
  )

amp_sum <- amp2 %>%
  group_by(EVO, Absent_norm, Timepoint) %>%
  summarise(mean_ci_95(amp_scaled), .groups = "drop") %>%
  transmute(
    EVO, Absent_norm, Timepoint,
    Series = lab_Camp,
    y = mean, ylo = pmax(lo, 0), yhi = pmin(hi, 1)
  )

plotB_df <- bind_rows(amp_sum, acav_sum) %>%
  mutate(
    EVO = factor(EVO, levels = c("ANC", "EVO")),
    Absent_norm = factor(Absent_norm, levels = absent_levels),
    Series = factor(Series, levels = c(lab_Camp, lab_acav_prop))
  )

# labels used in legends
lab_Camp <- "C<sub>amp</sub>"
lab_acav_prop <- "<i>A. caviae</i> proportion"

plotB_df <- bind_rows(amp_sum, acav_sum) %>%
  mutate(
    EVO = factor(EVO, levels = c("ANC", "EVO")),
    Absent_norm = factor(Absent_norm, levels = absent_levels),
    Series = factor(Series, levels = c(lab_Camp, lab_acav_prop))
  )

pB <- ggplot() +
  geom_tile(
    data = bg_tile_B,
    aes(x = Timepoint, y = y),
    inherit.aes = FALSE,
    fill = bg_present_fill,
    width = Inf, height = Inf
  ) +
  geom_point(
    data = bg_legend_key,
    aes(x = Timepoint, y = y, fill = bg_key),
    inherit.aes = FALSE,
    shape = 22, size = 4, colour = NA, alpha = 0.9,
    show.legend = TRUE
  ) +
  scale_fill_manual(values = setNames(bg_present_fill, bg_legend_label), name = NULL) +
  geom_errorbar(
    data = plotB_df,
    aes(x = Timepoint, y = y, ymin = ylo, ymax = yhi, colour = Series, group = Series),
    width = 0.35, linewidth = 0.9, na.rm = TRUE
  ) +
  geom_line(
    data = plotB_df,
    aes(x = Timepoint, y = y, colour = Series, linetype = Series, group = Series),
    linewidth = 1.0, na.rm = TRUE
  ) +
  geom_point(
    data = plotB_df,
    aes(x = Timepoint, y = y, colour = Series, group = Series),
    size = 3.2, na.rm = TRUE
  ) +
  scale_x_continuous(breaks = valid_times) +
  scale_y_continuous(
    limits = c(0, 1),
    labels = percent_format(accuracy = 1),
    sec.axis = sec_axis(~ . * conc_max, name = "Ampicillin (µg mL⁻¹)")
  ) +
  scale_colour_manual(
    values = setNames(
      c(col_amp, col_growth),
      c(lab_Camp, lab_acav_prop)
    ),
    breaks = c(lab_Camp, lab_acav_prop),
    labels = c(lab_Camp, lab_acav_prop)
  ) +
  scale_linetype_manual(
    values = setNames(
      c("dashed", "solid"),
      c(lab_Camp, lab_acav_prop)
    ),
    breaks = c(lab_Camp, lab_acav_prop),
    labels = c(lab_Camp, lab_acav_prop)
  ) +
  facet_grid(
    rows = vars(EVO),
    cols = vars(Absent_norm),
    labeller = labeller(
      EVO         = as_labeller(evo_labels_2line),
      Absent_norm = as_labeller(absent_labels_md)
    )
  ) +
  labs(
    x = "Time (h)",
    y = "<b><i>A. caviae</i> proportion</b>",
    colour = NULL,
    linetype = NULL,
    tag = "B"
  ) +
  guides(
    fill = guide_legend(order = 1, override.aes = list(shape = 22, size = 5, colour = NULL)),
    colour = guide_legend(order = 2),
    linetype = guide_legend(order = 2)
  ) +
  theme_bw(base_size = BASE_SIZE) +
  theme(
    panel.grid = element_blank(),
    panel.background = element_rect(fill = bg_base_fill, colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    strip.background = element_blank(),
    strip.text.x = ggtext::element_markdown(size = BASE_SIZE),
    strip.text.y = ggtext::element_markdown(size = BASE_SIZE),
    axis.title.x = element_text(size = BASE_SIZE, face = "bold"),
    axis.title.y = ggtext::element_markdown(size = BASE_SIZE, face = "bold"),
    axis.text    = element_text(size = BASE_SIZE),
    plot.tag     = element_text(size = BASE_SIZE + 2, face = "bold"),
    legend.position = "bottom",
    legend.box = "horizontal",
    legend.text = ggtext::element_markdown(size = BASE_SIZE),
    legend.spacing.x = unit(0.8, "cm")
  )


New names:
• `Column` -> `Column...3`
• `` -> `...8`
• `Column` -> `Column...11`


### Panel C

In [5]:
assign_day <- function(sample) {
  if (str_detect(sample, "d4"))  return(4)
  if (str_detect(sample, "d8"))  return(8)
  if (str_detect(sample, "d12")) return(12)
  if (str_detect(sample, "d16")) return(16)
  if (str_detect(sample, "d20")) return(20)
  NA_real_
}

assign_evo_orig <- function(sample) {
  if (str_starts(sample, "a_")) return("All ancestral")
  if (str_starts(sample, "d_")) return("Evolved 1972")
  if (str_starts(sample, "e_")) return("Evolved 1977")
  if (str_starts(sample, "f_")) return("All evolved")
  NA_character_
}

assign_replicate <- function(sample) {
  for (r in 1:4) if (str_detect(sample, paste0("_", r, "_"))) return(r)
  NA_real_
}

assign_AB <- function(sample) {
  if (str_detect(sample, "_t1_")) return(5)
  if (str_detect(sample, "_t2_")) return(50)
  if (str_detect(sample, "_t3_")) return(500)
  if (str_detect(sample, "_t4_")) return(0)
  NA_real_
}

species_raw_main <- read_tsv(file.path(data_dir, "species_abundance.txt"), show_col_types = FALSE) %>% fix_names()

meta_main <- species_raw_main %>%
  distinct(sample) %>%
  mutate(
    Day       = map_dbl(sample, assign_day),
    Evo_orig  = map_chr(sample, assign_evo_orig),
    Replicate = map_dbl(sample, assign_replicate),
    AB        = map_dbl(sample, assign_AB)
  ) %>%
  drop_na(Day, Evo_orig, Replicate, AB) %>%
  mutate(
    Day = as.integer(Day),
    Replicate = as.character(as.integer(Replicate)),
    AB = as.integer(AB),
    AB_fac = factor(as.character(AB), levels = c("0", "5", "50", "500"))
  )

species_main <- species_raw_main %>%
  inner_join(meta_main, by = "sample") %>%
  mutate(is_acav = (strainID == id_acav))

non_acav_df <- species_main %>%
  group_by(sample, Day, Evo_orig, AB, AB_fac, Replicate) %>%
  summarise(
    non_acav_sum = sum(f_correct[!is_acav], na.rm = TRUE),
    total_sum    = sum(f_correct, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(non_acav_frac = non_acav_sum / pmax(total_sum, 1e-12))

non_acav_change_wide <- non_acav_df %>%
  filter(Day %in% c(12, 16)) %>%
  group_by(Evo_orig, AB, AB_fac, Replicate, Day) %>%
  summarise(non_acav_frac = mean(non_acav_frac, na.rm = TRUE), .groups = "drop") %>%
  pivot_wider(names_from = Day, values_from = non_acav_frac) %>%
  drop_na(`12`, `16`) %>%
  mutate(delta_non_acav = `16` - `12`) %>%
  filter(Evo_orig %in% c("All ancestral", "All evolved")) %>%
  mutate(
    Evo_fac = recode(Evo_orig,
      "All ancestral" = "Anc. comm.",
      "All evolved"   = "Full primed"
    ),
    Evo_fac = factor(Evo_fac, levels = c("Anc. comm.", "Full primed"))
  )

yminC <- min(non_acav_change_wide$delta_non_acav, na.rm = TRUE)
ymaxC <- max(non_acav_change_wide$delta_non_acav, na.rm = TRUE)
padC  <- 0.05 * (ymaxC - yminC + 1e-9)

bg_tile_C <- tibble(
  Evo_fac = "Full primed",
  y = (yminC + ymaxC) / 2,
  height = (ymaxC - yminC) + 2 * padC
)

pC <- ggplot(non_acav_change_wide, aes(x = Evo_fac, y = delta_non_acav)) +
  scale_x_discrete(limits = c("Anc. comm.", "Full primed")) +
  geom_tile(
    data = bg_tile_C,
    aes(x = Evo_fac, y = y),
    inherit.aes = FALSE,
    fill = bg_present_fill,
    width = 0.95,
    height = bg_tile_C$height
  ) +
  geom_boxplot(aes(fill = AB_fac), outlier.shape = NA, linewidth = 0.7) +
  geom_point(
    aes(colour = AB_fac),
    position = position_jitterdodge(jitter.width = 0.10, dodge.width = 0.75),
    alpha = 0.7,
    size = 3.2,
    stroke = 0.25
  ) +
  scale_fill_manual(
    values = ab_cols,
    name = title_prepulse_Camp,
    labels = c("0", "5", "50", "500")
  ) +
  scale_colour_manual(values = ab_cols, guide = "none") +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  labs(
    x = NULL,
    y = "<b>Δ non-<i>A. caviae</i> fraction<br>during main ampicillin pulse</b>",
    tag = "C"
  ) +
  theme_bw(base_size = BASE_SIZE) +
  theme(
    panel.grid = element_blank(),
    panel.background = element_rect(fill = bg_base_fill, colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    axis.title.y = ggtext::element_markdown(size = BASE_SIZE, face = "bold"),
    axis.text.x  = element_text(size = BASE_SIZE),
    axis.text.y  = element_text(size = BASE_SIZE),
    legend.title = ggtext::element_markdown(size = BASE_SIZE),
    legend.text  = element_text(size = BASE_SIZE),
    legend.position = "right",
    plot.tag = element_text(size = BASE_SIZE + 2, face = "bold")
  )

pC <- pC + theme(
  plot.margin = margin(4, 2, 4, 2),
  legend.title = ggtext::element_markdown(size = BASE_SIZE - 1),
  legend.text  = element_text(size = BASE_SIZE - 1)
)

### Combine A+B and save Fig4

In [6]:
# left column remains stacked with patchwork
left_col <- (legend_top_patch / pA / pB) +
  plot_layout(heights = c(0.10, 1.18, 1.42))

# now combine LEFT and C with cowplot, which scales full grobs
combined <- cowplot::plot_grid(
  left_col,
  pC,
  nrow = 1,
  rel_widths = c(4, 2),
  align = "h",
  axis = "tb"
)

ggsave(out_main, combined, width = 23, height = 13.2, units = "in", device = cairo_pdf)
message("Saved combined A+B+C figure to: ", out_main)

Warning message:
“Removed 10 rows containing missing values or values outside the scale range
(`geom_point()`).”
Saved combined A+B+C figure to: ../figs/Fig5.pdf



### Supplementary figure: stacked areas + strong dashed amp with halo

In [7]:
counts_sum <- counts_filtered %>%
  group_by(EVO, Absent_norm, replicate, Timepoint, strainID) %>%
  summarise(f_sum = sum(f_correct, na.rm = TRUE), .groups = "drop") %>%
  group_by(EVO, Absent_norm, replicate, Timepoint) %>%
  mutate(
    frac_norm = f_sum / pmax(sum(f_sum, na.rm = TRUE), 1e-12),
    sqrt_val  = sqrt(frac_norm),
    y_comp    = sqrt_val / pmax(sum(sqrt_val, na.rm = TRUE), 1e-12)
  ) %>%
  ungroup() %>%
  mutate(
    EVO = factor(EVO, levels = c("ANC", "EVO")),
    Absent_norm = factor(Absent_norm, levels = absent_levels),
    replicate = factor(replicate),
    strainID = factor(strainID, levels = names(species_colors))
  )

amp_full <- amp2 %>%
  mutate(
    EVO = factor(EVO, levels = c("ANC", "EVO")),
    Absent_norm = factor(Absent_norm, levels = absent_levels),
    replicate = factor(replicate)
  )

pSupp <- ggplot(counts_sum, aes(x = Timepoint, y = y_comp, fill = strainID, group = strainID)) +
  geom_area(position = "stack", colour = NA) +
  geom_line(
    data = amp_full,
    aes(x = Timepoint, y = amp_scaled, group = 1),
    inherit.aes = FALSE,
    colour = "white",
    linewidth = 3.0,
    lineend = "round"
  ) +
  geom_line(
    data = amp_full,
    aes(x = Timepoint, y = amp_scaled, group = 1),
    inherit.aes = FALSE,
    colour = col_amp,
    linewidth = 2.0,
    linetype = "dashed",
    lineend = "round"
  ) +
  ggh4x::facet_grid2(
    rows = vars(Absent_norm),
    cols = vars(EVO, replicate),
    strip = ggh4x::strip_nested(bleed = TRUE),
    labeller = labeller(
      EVO = as_labeller(c(
        "ANC" = "Ancestral community",
        "EVO" = "Full primed ommunity"
      )),
      Absent_norm = as_labeller(absent_labels_md)
    )
  ) +
  scale_fill_manual(
    values = species_colors,
    breaks = names(species_labels_1line),
    labels = species_labels_1line,
    guide  = guide_legend(ncol = 1)
  ) +
  scale_x_continuous(breaks = valid_times) +
  scale_y_continuous(
    limits = c(0, 1.03),
    labels = percent_format(accuracy = 1),
    sec.axis = sec_axis(~ . * conc_max, name = "Ampicillin (µg mL⁻¹)")
  ) +
  labs(
    x = "Time (h)",
    y = "Relative abundance (sqrt transformed)",
    fill = NULL
  ) +
  theme_bw(base_size = 12) +
  theme(
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    panel.grid = element_blank(),
    strip.background = element_blank(),
    strip.text.x       = ggtext::element_markdown(size = 11),
    strip.text.y       = ggtext::element_markdown(size = 11),
    strip.text.y.left  = ggtext::element_markdown(size = 11),
    strip.text.y.right = ggtext::element_markdown(size = 11),
    legend.position = "right",
    legend.title = element_blank(),
    legend.text = ggtext::element_markdown(size = 14)
  )

ggsave(out_supp, pSupp, width = 20, height = 11, units = "in", device = cairo_pdf)
message("Saved supplementary figure to: ", out_supp)

Saved supplementary figure to: ../figs/ED_Fig5.pdf



### Analysis

In [8]:
suppressPackageStartupMessages({
  library(survival)
  library(broom)
  library(glue)
})

# ## Analysis settings + helpers

# Inputs 
counts_file <- file.path(data_dir, "species_abundance_loo.txt")
meta_file   <- file.path(data_dir, "metadata_loo.txt")
amp_file    <- file.path(data_dir, "AMP_conc_loo.txt")

threshold_abs <- 50  # µg/mL absolute depletion threshold
tau_rmst <- 24       # hours, RMST truncation (for RMST)

fmt_num <- function(x, d=2) if (is.na(x)) "NA" else formatC(x, format="f", digits=d)
fmt_p   <- function(p) { if (is.na(p)) "NA" else if (p < 1e-3) "<0.001" else formatC(p, format="f", digits=3) }
fmt_df <- function(df) {
  if (is.na(df)) return("NA")
  if (abs(df - round(df)) < 1e-6) sprintf("%d", round(df)) else formatC(df, format="f", digits=1)
}
fmt_tstat <- function(tval, df) glue("t_{fmt_df(df)} = {fmt_num(tval,2)}")

# RMST (τ) from KM, manual fallback
rmst_manual <- function(time, event, tau) {
  if (length(time) == 0) return(NA_real_)
  sf <- survfit(Surv(time, event) ~ 1)
  idx <- which(sf$time < tau)
  tt <- c(0, sf$time[idx], tau)
  S  <- c(1, sf$surv[idx])
  if (length(S) != length(tt) - 1) S <- c(S, rep(tail(S,1), (length(tt)-1) - length(S)))
  sum((tt[-1] - tt[-length(tt)]) * S)
}

boot_rmst_diff <- function(df, tau, B = 2000L, seed = 123L) {
  set.seed(seed)
  evo_reps <- unique(df$replicate[df$EVO == "EVO"])
  anc_reps <- unique(df$replicate[df$EVO == "ANC"])
  if (!length(evo_reps) || !length(anc_reps)) return(list(diff=NA_real_, ci=c(NA,NA), p=NA_real_))

  point <- rmst_manual(df$time[df$EVO=="EVO"], df$event[df$EVO=="EVO"], tau) -
    rmst_manual(df$time[df$EVO=="ANC"], df$event[df$EVO=="ANC"], tau)

  diffs <- rep(NA_real_, B)
  for (b in seq_len(B)) {
    evo_sel <- sample(evo_reps, length(evo_reps), replace = TRUE)
    anc_sel <- sample(anc_reps, length(anc_reps), replace = TRUE)
    evo_df <- df[df$EVO=="EVO" & df$replicate %in% evo_sel, , drop = FALSE]
    anc_df <- df[df$EVO=="ANC" & df$replicate %in% anc_sel, , drop = FALSE]
    if (nrow(evo_df) == 0 || nrow(anc_df) == 0) next
    diffs[b] <- rmst_manual(evo_df$time, evo_df$event, tau) -
      rmst_manual(anc_df$time, anc_df$event, tau)
  }
  diffs <- diffs[is.finite(diffs)]
  if (!length(diffs)) return(list(diff=point, ci=c(NA,NA), p=NA_real_))
  ci <- quantile(diffs, c(0.025, 0.975), na.rm = TRUE)
  p  <- 2 * min(mean(diffs >= 0), mean(diffs <= 0))
  list(diff = point, ci = ci, p = p)
}

t_one_with_twosided_CI <- function(x, mu = 0, alt = c("greater","less","two.sided")) {
  alt <- match.arg(alt)
  x <- x[is.finite(x)]
  if (length(x) < 2) return(list(mean=NA, ci=c(NA,NA), t=NA, df=NA, p=NA))
  tt_ci  <- t.test(x, mu = mu, alternative = "two.sided")
  tt <- if (alt == "two.sided") tt_ci else t.test(x, mu = mu, alternative = alt)
  list(mean = mean(x),
       ci   = unname(tt_ci$conf.int),
       t    = unname(tt$statistic),
       df   = unname(tt$parameter),
       p    = tt$p.value)
}

# ## Load APEX data for inference (same exclusions as figure)

counts <- fix_names(read.delim(counts_file, check.names = FALSE))
meta   <- fix_names(read.delim(meta_file,   check.names = FALSE))
amp    <- fix_names(read.delim(amp_file,    check.names = FALSE))

meta2_inf <- meta %>%
  mutate(Absent_norm = dplyr::recode(Absent, "All" = "All3", .default = Absent)) %>%
  filter(EVO %in% c("ANC","EVO"),
         !row %in% c("Cont","Neg","CC"),
         ampicillin_ug_ml == 500) %>%
  mutate(EVO = as.character(EVO),
         Absent_norm = as.character(Absent_norm),
         replicate   = as.character(replicate))

# Composition (untransformed for inference)
comp0 <- counts %>%
  inner_join(meta2_inf %>% select(sample, Timepoint, EVO, replicate, Absent_norm), by = "sample") %>%
  filter(
    !(Absent_norm == "1972" & strainID == id_acav),
    !(Absent_norm == "1977" & strainID == id_1977),
    !(Absent_norm == "403"  & strainID == id_0403),
    !(Absent_norm == "All3" & strainID %in% degrader_ids)
  ) %>%
  group_by(EVO, Absent_norm, replicate, Timepoint, strainID) %>%
  summarise(count = sum(f_correct, na.rm = TRUE), .groups = "drop") %>%
  group_by(EVO, Absent_norm, replicate, Timepoint) %>%
  mutate(frac = count / pmax(sum(count), 1e-12)) %>%
  ungroup()

acav <- comp0 %>%
  filter(strainID == id_acav) %>%
  transmute(EVO, Absent_norm, replicate, Timepoint, Aeromonas = frac)

nonfocal <- comp0 %>%
  group_by(EVO, Absent_norm, replicate, Timepoint) %>%
  summarise(nonfocal = 1 - sum(frac[strainID == id_acav], na.rm = TRUE), .groups = "drop")

# Ampicillin time-courses
amp_rep <- amp %>%
  filter(Evo %in% c("ANC","EVO")) %>%
  mutate(Absent_norm = dplyr::recode(Missing_species, "none" = "None", .default = Missing_species),
         EVO       = as.character(Evo),
         replicate = as.character(Rep),
         Timepoint = Time_h,
         Conc_ug_ml = C_amp) %>%
  select(EVO, Absent_norm, replicate, Timepoint, Conc_ug_ml)

# Event times (≤ 50 µg/mL)
tdeg <- amp_rep %>%
  group_by(EVO, Absent_norm, replicate) %>%
  summarise(
    t_star = { ix <- which(Conc_ug_ml <= threshold_abs)
    if (length(ix) == 0) NA_real_ else min(Timepoint[ix], na.rm = TRUE) },
    t_max = max(Timepoint, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(event         = !is.na(t_star),
         time          = ifelse(event, t_star, t_max),
         A1972_present = ifelse(Absent_norm %in% c("None","403","1977"), "Present", "Absent"))

tdeg_pres <- tdeg %>% filter(A1972_present == "Present")

# ## 1) Time-to-≤50 µg/mL: log-rank + RMST(τ=24 h)

surv_obj <- Surv(time = tdeg_pres$time, event = tdeg_pres$event)
grp <- factor(tdeg_pres$EVO, levels = c("ANC","EVO"),
              labels = c("Ancestral", "Ampicillin pre-exposed"))

lr <- survdiff(surv_obj ~ grp)
lr_chisq <- lr$chisq
lr_p <- pchisq(lr_chisq, df = length(lr$n) - 1, lower.tail = FALSE)

# RMST: use survRM2 if available, else bootstrap fallback
if (requireNamespace("survRM2", quietly = TRUE)) {
  df_rmst <- tdeg_pres %>% transmute(time = time, event = as.integer(event), arm = ifelse(EVO=="EVO",1,0))
  rm2 <- survRM2::rmst2(time = df_rmst$time, status = df_rmst$event, arm = df_rmst$arm, tau = tau_rmst)
  rmst_diff <- rm2$unadjusted.result["RMST (arm=1)-(arm=0)", "Est."]
  rmst_ci   <- c(rm2$unadjusted.result["RMST (arm=1)-(arm=0)", "Lower .95"],
                 rm2$unadjusted.result["RMST (arm=1)-(arm=0)", "Upper .95"])
  rmst_p    <- rm2$unadjusted.result["RMST (arm=1)-(arm=0)", "p"]
} else {
  rm_boot <- boot_rmst_diff(tdeg_pres %>% select(EVO, time, event, replicate), tau = tau_rmst, B = 2000L, seed = 123)
  rmst_diff <- rm_boot$diff; rmst_ci <- rm_boot$ci; rmst_p <- rm_boot$p
}

cat("\n[1] Time-to-≤", threshold_abs, " µg/mL (A. caviae present panels)\n", sep = "")
cat("  Log-rank: chi^2 = ", fmt_num(lr_chisq,2), ", p = ", fmt_p(lr_p), "\n", sep = "")
cat("  RMST diff (EVO-ANC), tau=24h: ", fmt_num(rmst_diff,2),
    " h (95% CI ", fmt_num(rmst_ci[1],2), " to ", fmt_num(rmst_ci[2],2),
    "), p = ", fmt_p(rmst_p), "\n", sep = "")

# ## 2) Post-depletion decline of A. caviae (pre-exposed only; slope after t*)

acav_evo_slopes <- acav %>%
  inner_join(tdeg %>% filter(EVO == "EVO", event) %>% select(EVO, Absent_norm, replicate, t_star),
             by = c("EVO","Absent_norm","replicate")) %>%
  group_by(Absent_norm, replicate) %>%
  group_modify(~{
    dd <- .x %>% filter(Timepoint >= t_star)
    if (nrow(dd) >= 2) tibble(slope = coef(lm(Aeromonas ~ Timepoint, data = dd))[["Timepoint"]])
    else tibble(slope = NA_real_)
  }) %>%
  ungroup() %>%
  drop_na(slope)

acav_evo_stat <- if (nrow(acav_evo_slopes) >= 2)
  t_one_with_twosided_CI(acav_evo_slopes$slope, mu = 0, alt = "less") else
    list(mean=NA, ci=c(NA,NA), t=NA, df=NA, p=NA)

cat("\n[2] Post-depletion A. caviae slope (pre-exposed only; after t*)\n")
cat("  Mean slope = ", fmt_num(acav_evo_stat$mean, 4),
    " (95% CI ", fmt_num(acav_evo_stat$ci[1],4), " to ", fmt_num(acav_evo_stat$ci[2],4), ")\n", sep = "")
cat("  One-sided test (less than 0): ", fmt_tstat(acav_evo_stat$t, acav_evo_stat$df),
    ", p = ", fmt_p(acav_evo_stat$p), "\n", sep = "")

# ## 3) Directional slopes 6–24 h:
# ###    - A. caviae in ancestral: > 0
# ###    - non-Aeromonas in pre-exposed: > 0
# ###    - non-Aeromonas in ancestral: < 0
# ###    - between-groups comparison (Welch + Wilcoxon)

# A. caviae slope 6–24 h in ancestral
acav_anc_slope <- acav %>%
  inner_join(tdeg %>% select(EVO, Absent_norm, replicate, A1972_present),
             by = c("EVO","Absent_norm","replicate")) %>%
  filter(A1972_present == "Present", EVO == "ANC", Timepoint >= 6, Timepoint <= 24) %>%
  group_by(Absent_norm, replicate) %>%
  summarise(
    slope_6_24 = {
      df <- dplyr::pick(Timepoint, Aeromonas)
      if (nrow(df) >= 2) coef(lm(Aeromonas ~ Timepoint, data = df))[["Timepoint"]] else NA_real_
    },
    .groups = "drop"
  ) %>%
  drop_na(slope_6_24)

acav_anc_stat <- if (nrow(acav_anc_slope) >= 2)
  t_one_with_twosided_CI(acav_anc_slope$slope_6_24, mu = 0, alt = "greater") else
    list(mean=NA, ci=c(NA,NA), t=NA, df=NA, p=NA)

# non-Aeromonas slopes 6–24 h
nf_slope <- nonfocal %>%
  inner_join(tdeg %>% select(EVO, Absent_norm, replicate, A1972_present),
             by = c("EVO","Absent_norm","replicate")) %>%
  filter(A1972_present == "Present", Timepoint >= 6, Timepoint <= 24) %>%
  group_by(EVO, Absent_norm, replicate) %>%
  summarise(
    slope_6_24 = {
      df <- dplyr::pick(Timepoint, nonfocal)
      if (nrow(df) >= 2) coef(lm(nonfocal ~ Timepoint, data = df))[["Timepoint"]] else NA_real_
    },
    .groups = "drop"
  ) %>%
  drop_na(slope_6_24)

nf_evo_stat <- if (sum(nf_slope$EVO=="EVO") >= 2)
  t_one_with_twosided_CI(nf_slope$slope_6_24[nf_slope$EVO=="EVO"], mu = 0, alt = "greater") else
    list(mean=NA, ci=c(NA,NA), t=NA, df=NA, p=NA)

nf_anc_stat <- if (sum(nf_slope$EVO=="ANC") >= 2)
  t_one_with_twosided_CI(nf_slope$slope_6_24[nf_slope$EVO=="ANC"], mu = 0, alt = "less") else
    list(mean=NA, ci=c(NA,NA), t=NA, df=NA, p=NA)

t_slope <- if (n_distinct(nf_slope$EVO) == 2) t.test(slope_6_24 ~ EVO, data = nf_slope) else NULL
w_slope <- if (n_distinct(nf_slope$EVO) == 2) wilcox.test(slope_6_24 ~ EVO, data = nf_slope, exact = FALSE) else NULL

cat("\n[3] Slopes 6–24 h (A. caviae present panels)\n")
cat("  A. caviae slope (ancestral; >0): mean = ", fmt_num(acav_anc_stat$mean,4),
    " (95% CI ", fmt_num(acav_anc_stat$ci[1],4), " to ", fmt_num(acav_anc_stat$ci[2],4), "); ",
    fmt_tstat(acav_anc_stat$t, acav_anc_stat$df), ", p(one-sided) = ", fmt_p(acav_anc_stat$p), "\n", sep = "")

cat("  non-Aeromonas slope (pre-exposed; >0): mean = ", fmt_num(nf_evo_stat$mean,4),
    " (95% CI ", fmt_num(nf_evo_stat$ci[1],4), " to ", fmt_num(nf_evo_stat$ci[2],4), "); ",
    fmt_tstat(nf_evo_stat$t, nf_evo_stat$df), ", p(one-sided) = ", fmt_p(nf_evo_stat$p), "\n", sep = "")

cat("  non-Aeromonas slope (ancestral; <0): mean = ", fmt_num(nf_anc_stat$mean,4),
    " (95% CI ", fmt_num(nf_anc_stat$ci[1],4), " to ", fmt_num(nf_anc_stat$ci[2],4), "); ",
    fmt_tstat(nf_anc_stat$t, nf_anc_stat$df), ", p(one-sided) = ", fmt_p(nf_anc_stat$p), "\n", sep = "")

if (!is.null(t_slope)) {
  cat("  Between communities (non-Aeromonas slope): Welch ",
      fmt_tstat(unname(t_slope$statistic), unname(t_slope$parameter)),
      ", p = ", fmt_p(t_slope$p.value),
      "; Wilcoxon p = ", fmt_p(w_slope$p.value), "\n", sep = "")
} else {
  cat("  Between communities (non-Aeromonas slope): not defined (missing group).\n")
}

cat("\nDone.\n")


[1] Time-to-≤50 µg/mL (A. caviae present panels)
  Log-rank: chi^2 = 23.00, p = <0.001
  RMST diff (EVO-ANC), tau=24h: -12.00 h (95% CI -12.00 to -12.00), p = <0.001

[2] Post-depletion A. caviae slope (pre-exposed only; after t*)
  Mean slope = -0.0181 (95% CI -0.0203 to -0.0159)
  One-sided test (less than 0): t_11 = -18.35, p = <0.001

[3] Slopes 6–24 h (A. caviae present panels)
  A. caviae slope (ancestral; >0): mean = 0.0329 (95% CI 0.0232 to 0.0426); t_11 = 7.49, p(one-sided) = <0.001
  non-Aeromonas slope (pre-exposed; >0): mean = 0.0027 (95% CI 0.0004 to 0.0051); t_11 = 2.54, p(one-sided) = 0.014
  non-Aeromonas slope (ancestral; <0): mean = -0.0329 (95% CI -0.0426 to -0.0232); t_11 = -7.49, p(one-sided) = <0.001
  Between communities (non-Aeromonas slope): Welch t_12.3 = -7.87, p = <0.001; Wilcoxon p = <0.001

Done.


### Main experiment AMP degradation stats

In [9]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(nlme)
})

# ---- load main-experiment AMP data ----
df_amp_main <- read.table("../data/AMP_conc.txt", header = TRUE, sep = "\t") %>%
  filter(Evo != "No_BAC") %>%
  mutate(
    # ensure numeric for model terms
    AB  = as.numeric(AB),
    Day = as.numeric(Day),
    # match original correlation grouping (strip day suffix)
    Sample2 = gsub("_d.*", "", Sample)
  )

# ---- GLS with AR(1) within Sample2 (unchanged spec) ----
M1_amp_main_exp <- gls(
  log(AMP_conc) ~ Evo * AB * Day,
  data        = df_amp_main,
  correlation = corAR1(form = ~ 1 | Sample2),
  method      = "ML"
)

# ---- format & print ONLY the table (aov_M1_amp_main_exp) ----
aov_M1_amp_main_exp <- as.data.frame(anova(M1_amp_main_exp))

rownames(aov_M1_amp_main_exp) <- c(
  "(Intercept)",
  "Strain pre-exposure",
  "Pre-pulse",
  "Day",
  "Strain pre-exposure x pre-pulse",
  "Strain pre-exposure x day",
  "Pre-pulse x day",
  "Strain pre-exposure x pre-pulse x day"
)

aov_M1_amp_main_exp <- cbind(Factor = rownames(aov_M1_amp_main_exp), aov_M1_amp_main_exp)
colnames(aov_M1_amp_main_exp)[2] <- "DF"

print(aov_M1_amp_main_exp)

                                                                     Factor DF
(Intercept)                                                     (Intercept)  1
Strain pre-exposure                                     Strain pre-exposure  3
Pre-pulse                                                         Pre-pulse  1
Day                                                                     Day  1
Strain pre-exposure x pre-pulse             Strain pre-exposure x pre-pulse  3
Strain pre-exposure x day                         Strain pre-exposure x day  3
Pre-pulse x day                                             Pre-pulse x day  1
Strain pre-exposure x pre-pulse x day Strain pre-exposure x pre-pulse x day  3
                                          F-value      p-value
(Intercept)                           457.5151278 5.585836e-57
Strain pre-exposure                     0.5005883 6.822339e-01
Pre-pulse                               1.4762807 2.255819e-01
Day                                  

### Non-Aeromonas fraction analyses

In [10]:
# ================================================================
# MAIN EXPERIMENT: non-Aeromonas (non-HAMBI_1972) fraction analyses
#   - STATS ONLY: prints to screen; no files written; no plots
#   - Uses: ../data/species_abundance.txt (same as Fig4 Panel B)
# ================================================================

suppressPackageStartupMessages({
  library(tidyverse)
  library(broom)
})

species_raw_main <- read_tsv(file.path(data_dir, "species_abundance.txt"), show_col_types = FALSE) %>% fix_names()

# ---- reconstruct meta fields (same parsing logic as Panel B) ----
species_main2 <- species_raw_main %>%
  mutate(
    Day = case_when(
      str_detect(sample, "d4")  ~ 4,
      str_detect(sample, "d8")  ~ 8,
      str_detect(sample, "d12") ~ 12,
      str_detect(sample, "d16") ~ 16,
      str_detect(sample, "d20") ~ 20,
      TRUE ~ NA_real_
    ),
    Evo_orig = case_when(
      str_starts(sample, "a_") ~ "All ancestral",
      str_starts(sample, "d_") ~ "Evolved 1972",
      str_starts(sample, "e_") ~ "Evolved 1977",
      str_starts(sample, "f_") ~ "All evolved",
      TRUE ~ NA_character_
    ),
    Replicate = case_when(
      str_detect(sample, "_1_") ~ 1L,
      str_detect(sample, "_2_") ~ 2L,
      str_detect(sample, "_3_") ~ 3L,
      str_detect(sample, "_4_") ~ 4L,
      TRUE ~ NA_integer_
    ),
    AB = case_when(
      str_detect(sample, "_t1_") ~ 5L,
      str_detect(sample, "_t2_") ~ 50L,
      str_detect(sample, "_t3_") ~ 500L,
      str_detect(sample, "_t4_") ~ 0L,
      TRUE ~ NA_integer_
    )
  ) %>%
  filter(!is.na(Day), !is.na(Evo_orig), !is.na(AB), !is.na(Replicate)) %>%
  mutate(
    strainID  = gsub("-", "_", strainID),
    Replicate = factor(Replicate),
    AB_fac    = factor(AB, levels = c(0,5,50,500), labels = c("0","5","50","500"))
  )

# ---- compute non-Aeromonas fraction per sample ----
aero_ids <- c("HAMBI_1972")

non_aero_df <- species_main2 %>%
  mutate(is_aero = strainID %in% aero_ids) %>%
  group_by(sample, Day, Evo_orig, AB, AB_fac, Replicate) %>%
  summarise(
    non_aero_sum  = sum(f_correct[!is_aero], na.rm = TRUE),
    total_sum     = sum(f_correct, na.rm = TRUE),
    non_aero_frac = non_aero_sum / pmax(total_sum, 1e-12),
    .groups       = "drop"
  ) %>%
  mutate(
    PreExp_group = if_else(Evo_orig == "All evolved", "All pre-exposed", "Others"),
    PreExp_group = factor(PreExp_group, levels = c("Others", "All pre-exposed"))
  )

# ---- Δ from day 12 to day 16 (within Evo_orig × AB × Replicate) ----
non_aero_change <- non_aero_df %>%
  filter(Day %in% c(12, 16)) %>%
  group_by(Evo_orig, AB, AB_fac, Replicate, PreExp_group, Day) %>%
  summarise(non_aero_frac = mean(non_aero_frac, na.rm = TRUE), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = Day, values_from = non_aero_frac, names_prefix = "d") %>%
  filter(!is.na(d12), !is.na(d16)) %>%
  mutate(delta_non_aero = d16 - d12)

# ---- models ----
aov_nonAero_delta <- aov(delta_non_aero ~ Evo_orig * AB_fac, data = non_aero_change)
lm_nonAero_delta_PE <- lm(delta_non_aero ~ PreExp_group, data = non_aero_change)

# ---- PRINT STATS ONLY ----
cat("\n[Non-Aeromonas fraction analyses | MAIN experiment]\n")

cat("\n[1] ANOVA: Δ non-Aeromonas fraction (day16 - day12) ~ Evo_orig * AB_fac\n")
print(broom::tidy(aov_nonAero_delta))

#cat("\n[3] LM: Δ non-Aeromonas fraction ~ PreExp_group (All pre-exposed vs Others)\n")
print(broom::tidy(lm_nonAero_delta_PE))
cat("\nANOVA (LM):\n")
print(broom::tidy(anova(lm_nonAero_delta_PE)))

cat("\nDone.\n")


[Non-Aeromonas fraction analyses | MAIN experiment]

[1] ANOVA: Δ non-Aeromonas fraction (day16 - day12) ~ Evo_orig * AB_fac
# A tibble: 4 × 6
  term               df  sumsq   meansq statistic   p.value
  <chr>           <dbl>  <dbl>    <dbl>     <dbl>     <dbl>
1 Evo_orig            3 0.468  0.156       179.    4.58e-26
2 AB_fac              3 0.0200 0.00668       7.67  2.75e- 4
3 Evo_orig:AB_fac     9 0.0122 0.00135       1.55  1.57e- 1
4 Residuals          48 0.0418 0.000871     NA    NA       
# A tibble: 2 × 5
  term                        estimate std.error statistic  p.value
  <chr>                          <dbl>     <dbl>     <dbl>    <dbl>
1 (Intercept)                   -0.172   0.00544     -31.6 6.50e-40
2 PreExp_groupAll pre-exposed    0.195   0.0109       17.9 3.61e-26

ANOVA (LM):
# A tibble: 2 × 6
  term            df  sumsq  meansq statistic   p.value
  <chr>        <int>  <dbl>   <dbl>     <dbl>     <dbl>
1 PreExp_group     1 0.454  0.454        320.  3.61e-26
2 Resid

### Which non-A. caviae species were relatively favored during the main antibiotic

In [11]:
# ================================================================
# DESCRIPTIVE species decomposition of the non-A. caviae fraction
# during the main pulse
#
# Goal:
#   Qualitatively describe which species drive the difference between
#   "All ancestral" and "All evolved" communities.
#
# Metric:
#   species share within the non-A. caviae fraction
#
# Outputs:
#   [1] mean trajectories by species x group x day
#   [2] per-species day12 -> day16 change in All anc. vs All pre-exp.
#   [3] per-species trajectory slopes across days 12,16,20
#   [4] top species that increased in All pre-exp.
#   [5] species that declined in both, but declined less in All pre-exp.
#   [6] top species relatively favored in All pre-exp. vs All anc.
#   [7] top species relatively disfavored in All pre-exp. vs All anc.
#
# Notes:
#   - descriptive only; no per-species significance testing
#   - corrected so each species has ONE row with both anc + pre values
# ================================================================

suppressPackageStartupMessages({
  library(tidyverse)
})

# ----------------------------------------------------------------
# Inputs / helpers assumed already present in your script:
#   - data_dir
#   - fix_names()
#   - species_labels
# If not, define them before running this block.
# ----------------------------------------------------------------

id_acav <- "HAMBI_1972"

# ---- load main-experiment species table ----
species_raw_main <- read_tsv(
  file.path(data_dir, "species_abundance.txt"),
  show_col_types = FALSE
) %>%
  fix_names()

# ---- reconstruct metadata from sample names (same logic as Panel B) ----
species_main2 <- species_raw_main %>%
  mutate(
    Day = case_when(
      str_detect(sample, "d4")  ~ 4,
      str_detect(sample, "d8")  ~ 8,
      str_detect(sample, "d12") ~ 12,
      str_detect(sample, "d16") ~ 16,
      str_detect(sample, "d20") ~ 20,
      TRUE ~ NA_real_
    ),
    Evo_orig = case_when(
      str_starts(sample, "a_") ~ "All ancestral",
      str_starts(sample, "d_") ~ "Evolved 1972",
      str_starts(sample, "e_") ~ "Evolved 1977",
      str_starts(sample, "f_") ~ "All evolved",
      TRUE ~ NA_character_
    ),
    Replicate = case_when(
      str_detect(sample, "_1_") ~ 1L,
      str_detect(sample, "_2_") ~ 2L,
      str_detect(sample, "_3_") ~ 3L,
      str_detect(sample, "_4_") ~ 4L,
      TRUE ~ NA_integer_
    ),
    AB = case_when(
      str_detect(sample, "_t1_") ~ 5L,
      str_detect(sample, "_t2_") ~ 50L,
      str_detect(sample, "_t3_") ~ 500L,
      str_detect(sample, "_t4_") ~ 0L,
      TRUE ~ NA_integer_
    )
  ) %>%
  filter(!is.na(Day), !is.na(Evo_orig), !is.na(AB), !is.na(Replicate)) %>%
  mutate(
    strainID  = gsub("-", "_", strainID),
    Replicate = factor(Replicate),
    AB_fac    = factor(AB, levels = c(0, 5, 50, 500), labels = c("0", "5", "50", "500"))
  )

# ------------------------------------------------
# 1) Build species shares within non-A. caviae pool
# ------------------------------------------------
species_non_acav_traj <- species_main2 %>%
  filter(
    Evo_orig %in% c("All ancestral", "All evolved"),
    Day %in% c(12, 16, 20),
    strainID != id_acav
  ) %>%
  group_by(sample, Day, Evo_orig, AB, AB_fac, Replicate) %>%
  mutate(
    non_acav_total   = sum(f_correct, na.rm = TRUE),
    frac_in_non_acav = f_correct / pmax(non_acav_total, 1e-12)
  ) %>%
  ungroup() %>%
  mutate(
    Evo_fac = recode(
      Evo_orig,
      "All ancestral" = "All anc.",
      "All evolved"   = "All pre-exp."
    ),
    Evo_fac = factor(Evo_fac, levels = c("All anc.", "All pre-exp.")),
    Evo_fac_clean = recode(
      as.character(Evo_fac),
      "All anc."     = "anc",
      "All pre-exp." = "pre"
    )
  )

# ------------------------------------------------
# 2) Mean trajectory table by species x group x day
# ------------------------------------------------
species_day_summary <- species_non_acav_traj %>%
  group_by(strainID, Evo_fac, Day) %>%
  summarise(
    mean_frac = mean(frac_in_non_acav, na.rm = TRUE),
    sd_frac   = sd(frac_in_non_acav, na.rm = TRUE),
    n         = sum(is.finite(frac_in_non_acav)),
    se_frac   = sd_frac / sqrt(pmax(n, 1)),
    .groups   = "drop"
  ) %>%
  mutate(
    species_label = ifelse(
      strainID %in% names(species_labels),
      species_labels[strainID],
      strainID
    )
  ) %>%
  arrange(species_label, Evo_fac, Day)

cat("\n[1] Mean species trajectories within the non-A. caviae fraction]\n")
print(species_day_summary, n = Inf)

# ------------------------------------------------
# 3) Day12 -> Day16 change within each replicate/block
# ------------------------------------------------
species_delta_12_16 <- species_non_acav_traj %>%
  filter(Day %in% c(12, 16)) %>%
  group_by(Evo_fac_clean, strainID, AB, AB_fac, Replicate, Day) %>%
  summarise(
    frac_in_non_acav = sum(frac_in_non_acav, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  tidyr::pivot_wider(
    names_from = Day,
    values_from = frac_in_non_acav,
    names_prefix = "d"
  ) %>%
  filter(!is.na(d12), !is.na(d16)) %>%
  mutate(delta_12_16 = d16 - d12)

species_delta_summary <- species_delta_12_16 %>%
  group_by(strainID, Evo_fac_clean) %>%
  summarise(
    n                  = n(),
    mean_delta_12_16   = mean(delta_12_16, na.rm = TRUE),
    median_delta_12_16 = median(delta_12_16, na.rm = TRUE),
    min_delta_12_16    = min(delta_12_16, na.rm = TRUE),
    max_delta_12_16    = max(delta_12_16, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  tidyr::pivot_wider(
    names_from = Evo_fac_clean,
    values_from = c(
      n,
      mean_delta_12_16,
      median_delta_12_16,
      min_delta_12_16,
      max_delta_12_16
    ),
    names_sep = "_"
  ) %>%
  mutate(
    contrast_pre_minus_anc = mean_delta_12_16_pre - mean_delta_12_16_anc,
    species_label = ifelse(
      strainID %in% names(species_labels),
      species_labels[strainID],
      strainID
    )
  ) %>%
  select(
    strainID, species_label,
    n_anc, n_pre,
    mean_delta_12_16_anc, mean_delta_12_16_pre,
    median_delta_12_16_anc, median_delta_12_16_pre,
    min_delta_12_16_anc, min_delta_12_16_pre,
    max_delta_12_16_anc, max_delta_12_16_pre,
    contrast_pre_minus_anc
  ) %>%
  arrange(desc(contrast_pre_minus_anc))

cat("\n[2] Per-species day12 -> day16 change: All anc. vs All pre-exp.]\n")
print(species_delta_summary, n = Inf)

# ------------------------------------------------
# 4) Simple trajectory slopes across days 12,16,20
#    (descriptive only; no testing)
# ------------------------------------------------
species_slope_tbl <- species_non_acav_traj %>%
  group_by(Evo_fac_clean, strainID, AB, AB_fac, Replicate) %>%
  group_modify(~{
    dd <- .x %>%
      select(Day, frac_in_non_acav) %>%
      filter(is.finite(Day), is.finite(frac_in_non_acav)) %>%
      arrange(Day)

    if (nrow(dd) >= 2 && dplyr::n_distinct(dd$Day) >= 2) {
      tibble(
        slope_12_20 = coef(lm(frac_in_non_acav ~ Day, data = dd))[["Day"]]
      )
    } else {
      tibble(slope_12_20 = NA_real_)
    }
  }) %>%
  ungroup()

species_slope_summary <- species_slope_tbl %>%
  group_by(strainID, Evo_fac_clean) %>%
  summarise(
    n                  = sum(is.finite(slope_12_20)),
    mean_slope_12_20   = mean(slope_12_20, na.rm = TRUE),
    median_slope_12_20 = median(slope_12_20, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  tidyr::pivot_wider(
    names_from = Evo_fac_clean,
    values_from = c(n, mean_slope_12_20, median_slope_12_20),
    names_sep = "_"
  ) %>%
  mutate(
    slope_contrast_pre_minus_anc = mean_slope_12_20_pre - mean_slope_12_20_anc,
    species_label = ifelse(
      strainID %in% names(species_labels),
      species_labels[strainID],
      strainID
    )
  ) %>%
  select(
    strainID, species_label,
    n_anc, n_pre,
    mean_slope_12_20_anc, mean_slope_12_20_pre,
    median_slope_12_20_anc, median_slope_12_20_pre,
    slope_contrast_pre_minus_anc
  ) %>%
  arrange(desc(slope_contrast_pre_minus_anc))

cat("\n[3] Per-species trajectory slopes across days 12,16,20: All anc. vs All pre-exp.]\n")
print(species_slope_summary, n = Inf)

# ------------------------------------------------
# 5) Top species that increased in All pre-exp.
# ------------------------------------------------
top_increasing_preexp <- species_delta_summary %>%
  filter(is.finite(mean_delta_12_16_pre)) %>%
  filter(mean_delta_12_16_pre > 0) %>%
  arrange(desc(mean_delta_12_16_pre)) %>%
  select(
    strainID, species_label,
    mean_delta_12_16_anc,
    mean_delta_12_16_pre,
    contrast_pre_minus_anc,
    median_delta_12_16_anc,
    median_delta_12_16_pre
  )

cat("\n[4] Top species that increased in All pre-exp. (ranked by mean day12->16 change)]\n")
print(top_increasing_preexp, n = Inf)

# ------------------------------------------------
# 6) Species that declined in both, but declined less in pre-exp.
# ------------------------------------------------
top_declined_less_preexp <- species_delta_summary %>%
  filter(
    is.finite(mean_delta_12_16_anc),
    is.finite(mean_delta_12_16_pre),
    mean_delta_12_16_anc < 0,
    mean_delta_12_16_pre < 0,
    contrast_pre_minus_anc > 0
  ) %>%
  arrange(desc(contrast_pre_minus_anc)) %>%
  select(
    strainID, species_label,
    mean_delta_12_16_anc,
    mean_delta_12_16_pre,
    contrast_pre_minus_anc,
    median_delta_12_16_anc,
    median_delta_12_16_pre
  )

cat("\n[5] Species that declined in both groups, but declined less in All pre-exp.]\n")
print(top_declined_less_preexp, n = Inf)

# ------------------------------------------------
# 7) Top species relatively favored in pre-exp.
# ------------------------------------------------
top_favored_preexp <- species_delta_summary %>%
  filter(is.finite(contrast_pre_minus_anc)) %>%
  arrange(desc(contrast_pre_minus_anc)) %>%
  select(
    strainID, species_label,
    mean_delta_12_16_anc,
    mean_delta_12_16_pre,
    contrast_pre_minus_anc,
    median_delta_12_16_anc,
    median_delta_12_16_pre
  )

cat("\n[6] Top species relatively favored in All pre-exp. vs All anc.]\n")
print(top_favored_preexp, n = Inf)

# ------------------------------------------------
# 8) Species relatively disfavored in pre-exp.
# ------------------------------------------------
top_disfavored_preexp <- species_delta_summary %>%
  filter(is.finite(contrast_pre_minus_anc)) %>%
  arrange(contrast_pre_minus_anc) %>%
  select(
    strainID, species_label,
    mean_delta_12_16_anc,
    mean_delta_12_16_pre,
    contrast_pre_minus_anc,
    median_delta_12_16_anc,
    median_delta_12_16_pre
  )

cat("\n[7] Species relatively disfavored in All pre-exp. vs All anc.]\n")
print(top_disfavored_preexp, n = Inf)

cat("\nDone.\n")


[1] Mean species trajectories within the non-A. caviae fraction]
# A tibble: 132 × 8
    strainID   Evo_fac        Day mean_frac  sd_frac     n se_frac species_label
    <chr>      <fct>        <dbl>     <dbl>    <dbl> <int>   <dbl> <chr>        
  1 HAMBI_0097 All anc.        12 0          0          16 0       "Acinetobact…
  2 HAMBI_0097 All anc.        16 0          0          16 0       "Acinetobact…
  3 HAMBI_0097 All anc.        20 0          0          16 0       "Acinetobact…
  4 HAMBI_0097 All pre-exp.    12 0          0          16 0       "Acinetobact…
  5 HAMBI_0097 All pre-exp.    16 0          0          16 0       "Acinetobact…
  6 HAMBI_0097 All pre-exp.    20 0          0          16 0       "Acinetobact…
  7 HAMBI_0105 All anc.        12 0.0390     1.94e-2    16 4.84e-3 "Agrobacteri…
  8 HAMBI_0105 All anc.        16 0.0235     1.43e-2    16 3.57e-3 "Agrobacteri…
  9 HAMBI_0105 All anc.        20 0.0395     1.26e-2    16 3.14e-3 "Agrobacteri…
 10 HAMBI_0105 All pre-